#### Demo - Course Structured output
* Pydantic is a Python library that provides data validation and structured data modeling using Python type hints.

* Type hints describe expected types, while Pydantic uses those hints to actually validate and structure data.


* BaseModel is the core class in Pydantic. You create your own custom data models by subclassing BaseModel.

* The typing module in Python provides a way to add type hints to your code — so you can declare what kind of data your variables, function arguments, or class fields should have.

In [1]:
# import libraries
# Imports environment variables from a `.env` file.
from dotenv import load_dotenv

# Imports Agent, Runner, trace (for logging), and function_tool (for custom tools) from the agents module.
from agents import Agent, Runner, trace

# Importing the BaseModel class from the pydantic library, which is used to define data models with validation.

from pydantic import BaseModel
# Importing the List type hint from the typing module to specify lists of specific types
from typing import List

In [2]:

load_dotenv()

True

In [3]:

# Represents the detailed structure of a single course, including its title, description, learning outcomes, and modules.
class CourseDetail(BaseModel):
    coursetitle: str
    description: str
    learningoutcomes: List[str]
    coursemodules: List[str]


# Represents a collection of multiple courses, each described using the CourseDetail model.
class CourseDetailList(BaseModel):
    courses: List[CourseDetail]
   

In [4]:
instructions = """
You are an AI education specialist designing a university-level AI minor program using no-code tools.

Suggest exactly four beginner-friendly courses suitable for students from any academic background, including those with no prior experience in computer science or technology.

Each course spans eight weeks:
- Weeks 1-6: One module per week
- Weeks 7-8: Final project

Return the output as a JSON object with a key called 'courses', where the value is a list of 4 structured CourseDetail objects.

Each CourseDetail must contain:
- coursetitle: Title of the course
- description: Brief summary of the course
- learningoutcomes: 3-5 goals students will achieve
- coursemodules: List of weekly topics (6 learning modules + 2 weeks of final project)

Ensure the format strictly matches the required structure.
"""



In [5]:
course_agent = Agent(
    name="Course Details Agent",
    instructions=instructions,
    output_type=CourseDetailList,
    model="gpt-4o-mini"
)

In [6]:
prompt = "Design a beginner-friendly AI minor with four structured courses."

response = await Runner.run(course_agent, prompt)
course_list: CourseDetailList = response.final_output

In [7]:

print(course_list)

courses=[CourseDetail(coursetitle='Introduction to Artificial Intelligence', description='This course provides a foundational understanding of artificial intelligence, exploring its history, key concepts, and practical applications. Students will learn how AI impacts various fields and society as a whole.', learningoutcomes=['Understand the basic principles of AI and its history.', 'Identify various applications of AI across different industries.', 'Analyze the ethical implications of AI technologies.'], coursemodules=['What is AI? An Overview', 'History of AI: From Algorithms to Neural Networks', 'Types of AI: Reactive Machines to Self-Aware Systems', 'Applications of AI in Everyday Life', 'Ethical Considerations in AI Development', 'The Future of AI: Trends and Predictions', 'Final Project - Proposal and Research', 'Final Project - Presentation and Reflection']), CourseDetail(coursetitle='AI in Data Analysis', description='This course introduces students to the use of AI tools for da

In [8]:
print(response)

RunResult:
- Last agent: Agent(name="Course Details Agent", ...)
- Final output (CourseDetailList):
    {
      "courses": [
        {
          "coursetitle": "Introduction to Artificial Intelligence",
          "description": "This course provides a foundational understanding of artificial intelligence, exploring its history, basic concepts, and real-world applications.",
          "learningoutcomes": [
            "Understand the fundamental concepts of AI and its impact on society.",
            "Identify various applications of AI in different industries.",
            "Explore the ethical considerations surrounding AI technologies."
          ],
          "coursemodules": [
            "Week 1: The History of AI",
            "Week 2: Key Concepts and Terminology",
            "Week 3: AI Applications in Daily Life",
            "Week 4: Machine Learning Basics",
            "Week 5: Natural Language Processing (NLP)",
            "Week 6: Introduction to Neural Networks",
      

In [9]:
# Print in the nice format
for i, course in enumerate(course_list.courses, 1): # 1 is the start value
    print(f"\n Course {i}: {course.coursetitle}")
    print(f" Description: {course.description}")
    print(" Learning Outcomes:")
    for lo in course.learningoutcomes:
        print("-", lo)
    print("Modules:")
    for mod in course.coursemodules:
        print("-", mod)



 Course 1: Introduction to Artificial Intelligence
 Description: This course provides a foundational understanding of artificial intelligence, exploring its history, basic concepts, and real-world applications.
 Learning Outcomes:
- Understand the fundamental concepts of AI and its impact on society.
- Identify various applications of AI in different industries.
- Explore the ethical considerations surrounding AI technologies.
Modules:
- Week 1: The History of AI
- Week 2: Key Concepts and Terminology
- Week 3: AI Applications in Daily Life
- Week 4: Machine Learning Basics
- Week 5: Natural Language Processing (NLP)
- Week 6: Introduction to Neural Networks
- Week 7: Final Project Planning
- Week 8: Final Project Presentation

 Course 2: Data for AI: The Basics
 Description: This course delves into the critical role data plays in artificial intelligence, including data collection, analysis, and preparation for AI models.
 Learning Outcomes:
- Understand the types of data used in AI a

#### Why Structured Output?

Structured output gives you clean, machine-usable data, not just text blobs. That means you can:

* Feed the output into another agent (chaining agents)
* Store it in a database or curriculum system
* Visualize or display it in a UI
* Convert it to a file (CSV, PDF, JSON)
* Validate it programmatically (filter, sort, rank, customize)
* Trigger actions like sending an email, assigning a mentor, or building a course page

In [10]:
# Define the Marketing Copy Output Models
class CourseMarketing(BaseModel):
    coursetitle: str
    marketing_copy: str

class MarketingCopyList(BaseModel):
    marketing: List[CourseMarketing]


In [11]:
# Define the Marketing Agent

marketing_agent = Agent(
    name="Marketing Copy Agent",
    instructions=(
        "You are a university marketing assistant.\n"
        "You will receive a list of courses, each with a title and description.\n"
        "For each course, write a short, compelling marketing copy aimed at students from any background.\n"
        "Return the output in JSON with key 'marketing', each item containing 'coursetitle' and 'marketing_copy'."
    ),
    output_type=MarketingCopyList,
    model="gpt-4o"
)




In [12]:
# Get the list of courses
prompt = "Design a beginner-friendly AI minor with four structured courses."

response = await Runner.run(course_agent, prompt)
course_list: CourseDetailList = response.final_output

In [13]:
# Convert Your Structured Course List to a Prompt String
def format_course_list(course_list: CourseDetailList) -> str:
    prompt = "Here are the course titles and descriptions:\n\n"
    for course in course_list.courses:
        prompt += f"Course Title: {course.coursetitle}\n"
        prompt += f"Description: {course.description}\n\n"
    prompt += "Generate marketing copy for each course."
    return prompt

In [14]:
# Print the generated prompt
marketing_prompt = format_course_list(course_list)
print(marketing_prompt)



Here are the course titles and descriptions:

Course Title: Introduction to Artificial Intelligence
Description: This course provides an overview of AI concepts, applications, and ethical considerations. Students will explore how AI impacts various industries and society.

Course Title: Data Literacy for AI
Description: Students will learn how to collect, analyze, and interpret data, which is essential for building AI models. No prior experience with data is required.

Course Title: Machine Learning Basics
Description: This course introduces the fundamentals of machine learning. Students will explore supervised and unsupervised learning techniques with practical examples.

Course Title: AI and Society
Description: This course examines the social implications of AI technology, exploring its impact on jobs, privacy, and decision-making. Critical thinking and discussion are emphasized.

Generate marketing copy for each course.


In [15]:

# Run the Marketing Agent
marketing_response = await Runner.run(marketing_agent, marketing_prompt)
marketing_list: MarketingCopyList = marketing_response.final_output

In [17]:
# Print results
import dis
from IPython.display import Markdown


for promo in marketing_list.marketing:
    display(Markdown(promo.coursetitle))
   # print(f"\n{promo.coursetitle}")
    display(Markdown(promo.marketing_copy))
   # print(f"\n{promo.marketing_copy}")


Introduction to Artificial Intelligence

Step into the future with our Introduction to Artificial Intelligence course. Dive into the fascinating world of AI and discover how it's transforming industries and shaping society. No matter your background, this course opens the door to understanding the essential concepts and ethical considerations of AI today. Join us to become part of the next generation of innovators.

Data Literacy for AI

Unlock the power of data with our Data Literacy for AI course. Designed for all skill levels, you'll learn how to collect, analyze, and leverage data to fuel AI innovations. Perfect for beginners, this course empowers you with the knowledge needed to thrive in a data-driven world. Begin your journey into AI with confidence and curiosity.

Machine Learning Basics

Embark on your journey into the dynamic world of machine learning with our comprehensive Basics course. Gain hands-on experience with supervised and unsupervised techniques through practical, real-world examples. Whether you're starting fresh or enhancing your skills, this course is your gateway to developing innovative solutions with machine learning.

AI and Society

Explore the profound impact of AI on our world with our AI and Society course. Delve into critical discussions about jobs, privacy, and decision-making as you analyze the societal implications of this powerful technology. Ideal for students eager to engage in meaningful conversations and develop critical thinking skills, this course empowers you to make informed contributions to the AI landscape.

In [ ]:
# Strongly Typed Data - Type mismatch gives you an error

from pydantic import BaseModel
class Person(BaseModel):
    name: str
    age: int

person = Person(name="John", age=30)
# person = Person(name="John", age="thirty") # Gives you an error
print(person)


name='John' age=30
